In [ ]:
# Select Planet and Folder Structure
from typing import Literal
from pathlib import Path

import numpy as np
import ipywidgets
import matplotlib.pyplot as plt

from PIL import Image
from numpy.typing import NDArray
from matplotlib.gridspec import GridSpec

from mirage_texture_parser import MirageTextureLoader


def list_dir(folder: str | Path) -> list[Path]:
    """Scan a folder and return all folders inside."""
    folder = folder if isinstance(folder, Path) else Path(folder)
    return [f for f in folder.iterdir() if f.is_dir()]


def list_files(folder: str | Path, extension: str | None = None) -> list[Path]:
    """Scan a folder and return all files inside."""
    folder = folder if isinstance(folder, Path) else Path(folder)
    output_files = []
    for f in folder.iterdir():
        if f.is_file():
            if extension is not None:
                if f.suffix != extension:
                    continue
            output_files.append(f)
    return output_files


def detect_cubemap_bodies(folder: Path) -> dict[str, Path]:
    """Find body information for values within the folder."""
    bodies = {}

    def iter_search(fpath: Path):
        dirs = list_dir(fpath)
        if "Terrain" in [d.stem for d in dirs]:
            bodies[fpath.stem.rsplit("_", 1)[1]] = fpath / Path("Terrain")
            return
        for d in dirs:
            iter_search(d)

    iter_search(folder)
    return bodies


sol_dir = r"C:\Users\rweld\Documents\Kerbal Space Program 1\KSP Mirage Beta\GameData\Sol-Textures\PluginData"

bodies = {}
for f in list_dir(sol_dir):
    if f.name[0].isdigit():
        if int(f.name.split("_", 1)[0]):
            bodies.update(detect_cubemap_bodies(f))

select_widget = ipywidgets.Select(
    options=list(bodies.keys()),
    description="Select Body:",
    disabled=False,
)

select_widget

In [ ]:
# Define Cube Map Allocations
body_name = select_widget.value


class CubeMapPlanet:
    """Pull the files from Terrain to make a LOD cube map!"""

    def __init__(self, fpath: str | Path):
        self.folder = fpath if isinstance(fpath, Path) else Path(fpath)
        self.num_layers = len(list_dir(self.folder))

    def get_image(self, face: Literal["Xn", "Xp", "Yn", "Yp", "Zn", "Zp"]):
        """Pull image."""

    def project_pixel(
        self, lat: float, lon: float
    ) -> tuple[Literal["Xp", "Xn", "Yp", "Yn", "Zp", "Zn"], tuple[int, int]]:
        """Convert a lat / lon value into a pixel coordinate for the speficied face. Pixel location is (row, column) order with range [0, 1]."""

        lon = (lon + 360 + 180) % 360 - 180  # Wrap value for -180 < lon < 180
        lat = (lat + 180 + 90) % 180 - 90  # Wrap value for -90 < lat < 90

        rlon = lon * np.pi / 180  # To radians
        rlat = lat * np.pi / 180

        x = np.cos(rlon) * np.cos(rlat)
        y = np.sin(rlat)
        z = np.sin(rlon) * np.cos(rlat)

        a = max(np.abs(x), np.abs(y), np.abs(z))
        # Define u, v : -1 < u < 1 maps horizontal pixels and -1 < v < 1 maps vertical
        if a == np.abs(x):
            side = "Xp" if x > 0 else "Xn"  # 0 or 1
            u = -np.sign(x) * z / a
            v = -y / a
        elif a == np.abs(y):
            side = "Yp" if y > 0 else "Yn"  # 2 or 3
            u = x / a
            v = np.sign(y) * z / a
        else:
            side = "Zp" if z > 0 else "Zn"  # 4 or 5
            u = np.sign(z) * x / a
            v = -y / a

        row = (v + 1) / 2  # Vertical position in image - pixel row
        col = (u + 1) / 2  # Horizontal position in image - pixel column
        return side, (row, col)

    def stitch_image(
        self,
        face: Literal["Xn", "Xp", "Yn", "Yp", "Zn", "Zp"],
        lod: int = 0,
        image_type: Literal["colour", "height", "normal"] = "colour",
    ):
        """Get the combined image from the tiles in the texture."""
        assert lod < self.num_layers
        images = []

        textures = MirageTextureLoader(self.folder, lod)
        assert textures.colour_idx.num_entries == 6 * 4**lod  # 1, 4, 16 tex per face
        tex_per_face = textures.colour_idx.num_entries // 6
        tex_order = ["Xp", "Xn", "Yp", "Yn", "Zp", "Zn"]

        if lod == 0:
            return textures.get_texture(tex_order.index(face), image_type)

        order = [[0, 2], [1, 3]]
        indices = order
        for _ in range(lod - 1):
            size = len(indices) ** 2
            first_half = indices + [[x + size for x in segment] for segment in indices]
            last_half = [[x + 2 * size for x in segment] for segment in first_half]
            indices = [x[0] + x[1] for x in zip(first_half, last_half)]

        indices = [x for xs in indices for x in xs]

        for i in indices:
            im_array = textures.get_texture(
                tex_order.index(face) * tex_per_face + i, image_type
            )
            images.append(im_array)

        lines = []
        tile_count = 2**lod
        for i in range(tile_count):
            # Stitch while keeping padding in place
            imgs = images[tile_count * i : tile_count * (i + 1)]
            for j in range(tile_count):
                start = 0 if j == 0 else 4
                end = 264 if j == tile_count - 1 else 260
                imgs[j] = imgs[j][start:end, :, :]

            start = 0 if i == 0 else 4
            end = 264 if i == tile_count - 1 else 260
            imgs = [x[:, start:end, :] for x in imgs]  # Remove padding on middle layers
            res = np.concatenate(imgs, axis=0)
            lines.append(res)

        return np.concatenate(lines, axis=1)


dd = CubeMapPlanet(bodies[body_name])
data = dd.stitch_image("Yn", 3)

# Display Image
fig = plt.figure(figsize=(8, 6))
gs = GridSpec(3, 4, figure=fig, left=0, right=1, top=1, bottom=0)

layer = 3

xn = fig.add_subplot(gs[1, 0])
xn.imshow(dd.stitch_image("Xn", layer))
xn.set_aspect("equal")

zn = fig.add_subplot(gs[1, 1])
zn.imshow(dd.stitch_image("Zp", layer))
zn.set_aspect("equal")

yp = fig.add_subplot(gs[0, 1])
yp.imshow(dd.stitch_image("Yp", layer))
yp.set_aspect("equal")

yn = fig.add_subplot(gs[2, 1])
yn.imshow(dd.stitch_image("Yn", layer))
yn.set_aspect("equal")

xp = fig.add_subplot(gs[1, 2])
xp.imshow(dd.stitch_image("Xp", layer))
xp.set_aspect("equal")

zp = fig.add_subplot(gs[1, 3])
zp.imshow(dd.stitch_image("Zn", layer))
print(dd.stitch_image("Zp", layer).shape)
zp.set_aspect("equal")

fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
for ax in fig.get_axes():
    ax.set_xticks([])
    ax.set_yticks([])

plt.axis("off")
# Image.fromarray(data).save("image.png")
# plt.imshow(data)
plt.show()

In [ ]:
# Work out cube to rectangle projection
import cv2


def bilinear_interpolate(arr: NDArray, x: float, y: float) -> tuple[int, int, int, int]:
    """
    Interpolate a value at (x, y) in a 2D numpy array arr.
    """
    # 1. Identify surrounding grid indices
    x1, y1 = int(np.floor(x)), int(np.floor(y))
    x2, y2 = x1 + 1, y1 + 1

    # Boundary checks to prevent index errors
    x2 = min(x2, arr.shape[1] - 1)
    y2 = min(y2, arr.shape[0] - 1)

    # 2. Extract the four nearest neighbor values
    p11 = tuple(arr[y1, x1, :])  # Bottom-left
    p12 = tuple(arr[y2, x1, :])  # Top-left
    p21 = tuple(arr[y1, x2, :])  # Bottom-right
    p22 = tuple(arr[y2, x2, :])  # Top-right

    # 3. Calculate relative distances (weights)
    x_diff = x - x1
    y_diff = y - y1

    # 4. Compute weighted average
    # Formula: (1-dx)(1-dy)*p11 + (1-dx)*dy*p21 + dx*(1-dy)*p12 + dx*dy*p22
    res = []
    for channel in range(len(p11)):
        interpolated = (
            (1 - x_diff) * (1 - y_diff) * p11[channel]
            + (1 - x_diff) * y_diff * p21[channel]
            + x_diff * (1 - y_diff) * p12[channel]
            + x_diff * y_diff * p22[channel]
        )
        res.append(interpolated)

    return tuple(res)


def scansat_shading(
    colour_map: NDArray, normal_map: NDArray, normal_axis: int = 2
) -> NDArray:
    """Replicate the SCANsat shading code to see what it does."""
    hls_colours = cv2.cvtColor(colour_map, cv2.COLOR_RGB2HLS)

    opacity = 0.8

    lumOver = normal_map[:, :, normal_axis].astype(np.float32) / 255.0
    lum = hls_colours[:, :, 1].astype(np.float32) / 255.0

    new_lum = np.where(
        lum > 0.5,
        (opacity * (1 - (1 - (2 * (lumOver - 0.5))) * (1 - lum))) + (1 - opacity) * lum,
        (opacity * (2 * lumOver * lum)) + (1 - opacity) * lum,
    )

    hls_colours[:, :, 1] = (new_lum * 255.0).astype(np.int8)
    return cv2.cvtColor(hls_colours, cv2.COLOR_HLS2RGB)


def project_uv(cube: CubeMapPlanet, width: int, height: int) -> NDArray:
    """Cube Map to Equirectangular Projection."""

    w_scale = 360 / width
    h_scale = 180 / height

    lod = int(np.ceil(np.log(width) / np.log(2))) - 10
    print(lod)

    cube_sides: dict[str, NDArray] = {}
    for side in ["Xn", "Xp", "Yn", "Yp", "Zn", "Zp"]:
        colour_tile = cube.stitch_image(side, lod, "colour")  # type: ignore[reportTypeArgument]
        # cube_sides[side] = colour_tile
        normal_tile = cube.stitch_image(side, lod, "normal")  # type: ignore [reportTypeArgument]
        cube_sides[side] = scansat_shading(colour_tile, normal_tile, 1)

    pixel = np.zeros([height, width, cube_sides["Xp"].shape[2]], dtype=np.uint8)
    padding = 4  # Number of padded pixels around each image
    s = cube_sides["Xp"].shape[0] - 2 * padding

    for h_pixel in range(height):
        for w_pixel in range(width):
            lat = 90 - h_pixel * h_scale
            lon = w_pixel * w_scale - 180 + 20

            side, (row, col) = cube.project_pixel(lat, lon)
            # p = bilinear_interpolate(cube_sides[side], p2 + padding, p1 + padding)
            # pixel[h_pixel, w_pixel, :] = p
            row = int(round(s * row))
            col = int(round(s * col))
            pixel[h_pixel, w_pixel, :] = cube_sides[side][
                row + padding, col + padding, :
            ]

    return pixel


proj_data = project_uv(dd, 2048, 1024)
print(f"End data shape {proj_data.shape}")

# Display Image
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.axis("off")
Image.fromarray(proj_data).save("image.png")
plt.imshow(proj_data)
plt.show()

In [ ]:
class Bc7ModeInfo:
    def __init__(
        self,
        mode: int,
        num_subsets: int,
        partition_bits: int,
        rotation_bits: int,
        index_selection_bits: int,
        colour_bits: int,
        alpha_bits: int,
        endpoint_p_bits: int,
        shared_p_bits: int,
        index_bits: tuple[int, int],
    ) -> None:
        self.mode = mode
        self.num_subsets = num_subsets
        self.partition_bits = partition_bits
        self.rotation_bits = rotation_bits
        self.index_selection_bits = index_selection_bits
        self.colour_bits = colour_bits
        self.alpha_bits = alpha_bits
        self.endpoint_p_bits = endpoint_p_bits
        self.shared_p_bits = shared_p_bits
        self.index_bits = index_bits

    def to_data(self) -> int:
        """Return the number of bits before the data section."""
        return (
            self.partition_bits
            + self.rotation_bits
            + self.index_selection_bits
            + self.mode
            + 1
        )

    def get_pbits(self, bitblock: int):
        """Collect the required pbits for each things."""
        shifted = bitblock >> self.to_pbits()  # pbits now lowest bits
        num_bits = 2 * self.num_subsets * self.endpoint_p_bits + 2 * self.shared_p_bits
        return shifted & ((1 << num_bits) - 1)  # Mask out only num_bits

    def to_pbits(self) -> int:
        """Return the number of bits before the pbit definitions."""
        return self.to_data() + 2 * self.num_subsets * (
            3 * self.colour_bits + self.alpha_bits
        )


mode_info_list = [
    #              +---------------------------- num subsets
    #              |  +------------------------- partition bits
    #              |  |  +---------------------- rotation bits
    #              |  |  |  +------------------- index selection bits
    #              |  |  |  |  +---------------- color bits
    #              |  |  |  |  |  +------------- alpha bits
    #              |  |  |  |  |  |  +---------- endpoint P-bits
    #              |  |  |  |  |  |  |  +------- shared P-bits
    #              |  |  |  |  |  |  |  |    +-- 2x index bits
    Bc7ModeInfo(0, 3, 4, 0, 0, 4, 0, 1, 0, (3, 0)),  # 0
    Bc7ModeInfo(1, 2, 6, 0, 0, 6, 0, 0, 1, (3, 0)),  # 1
    Bc7ModeInfo(2, 3, 6, 0, 0, 5, 0, 0, 0, (2, 0)),  # 2
    Bc7ModeInfo(3, 2, 6, 0, 0, 7, 0, 1, 0, (2, 0)),  # 3
    Bc7ModeInfo(4, 1, 0, 2, 1, 5, 6, 0, 0, (2, 3)),  # 4
    Bc7ModeInfo(5, 1, 0, 2, 0, 7, 8, 0, 0, (2, 2)),  # 5
    Bc7ModeInfo(6, 1, 0, 0, 0, 7, 7, 1, 0, (4, 0)),  # 6
    Bc7ModeInfo(7, 2, 6, 0, 0, 5, 5, 1, 0, (2, 0)),  # 7
]


def get_num_subsets(mode: int) -> int:
    """Number of colour subsets per block - mode dependent."""
    return mode_info_list[mode].num_subsets


def get_partition_bits(mode: int) -> int:
    """Pixel Partition type bits - mode dependent."""
    return mode_info_list[mode].partition_bits


def get_rotation_bits(mode: int) -> int:
    """Rotation bits - used to rotate which colours are in which field."""
    return mode_info_list[mode].rotation_bits


def get_colour_bits(mode: int) -> int:
    """Return number of bits used to encode each colour."""
    return mode_info_list[mode].colour_bits


def get_alpha_bits(mode: int) -> int:
    """Return number of bits used to encode each alpha channel."""
    return mode_info_list[mode].alpha_bits


def extract_pbits(mode: int, bitblock: int) -> int:
    """Pull the pbits out of the bit block."""
    return mode_info_list[mode].get_pbits(bitblock)

In [ ]:
textures = MirageTextureLoader(dd.folder, 0)
tile = textures.colour_blob.get_tile(textures.colour_idx.get_idx(0))
dds_bytestream = tile.decodeTilePayload()  # BC7 bytestream

print(len(dds_bytestream))
print(", ".join([f"0x{x:02X}" for x in dds_bytestream[:16]]))


def extract_mode(bitblock: int) -> int:
    """Get BC7 mode for the block. Accepts bitblock of a 16-byte integer."""
    for i in range(8):
        if bitblock & (1 << i):
            return i
    raise ValueError("Mode cannot be decoded from byte value 0x00.")

def extract_partition_set_id(mode: int, bitblock: int):
    assert mode in [0, 1, 2, 3, 7]  # No partition in other modes
    # Drop the mode bits from the end of the field
    x = bitblock >> (mode + 1)
    if mode:  # Mode 0 uses 4-bit, others use 6-bit
        return x & 0x3F  # 6 bit partition
    return x & 0xF  # 4 bit partition

def get_partition_index(num_subsets: int, partition_set_id: int, x: int, y: int) -> int:
    """Does something to decode which partition the colour should be sampled from."""
    return 1  # TODO

def extract_endpoints(mode: int, bitblock: int) -> list[list[int]]:
    """Extract the compressed endpoint data from the block."""
    num_points = get_num_subsets(mode) * 2
    bpp = get_colour_bits(mode)
    colour_bitmask = (1 << bpp) - 1
    alpha_bits = get_alpha_bits(mode)
    alpha_bitmask = (1 << alpha_bits) - 1

    colours = []
    for i in range(num_points):
        r = bitblock >> (bpp * i) & colour_bitmask
        g = bitblock >> num_points * bpp >> (bpp * i) & colour_bitmask
        b = bitblock >> num_points * bpp * 2 >> (bpp * i) & colour_bitmask
        colour = [r, g, b, 255]  # Assume full opacity - not sure if this is correct
        if mode in [6, 7]:
            a = bitblock >> num_points * bpp * 3 >> (alpha_bits * i) & alpha_bitmask
            colour[3] = a
        colours.append(colour)

    return colours

def fully_decode_endpoints(endpoint_array: list[list[int]], mode: int, bitblock: int):
    """Decode end points??"""
    # First handle modes that have P-bits
    if mode in [0, 1, 3, 6, 7]:
        for i in range(len(endpoint_array)):
            # Component-wise left-shift
            for j in range(len(endpoint_array[i])):
                endpoint_array[i][j] = endpoint_array[i][j] << 1

        # Get all pbits
        pbits = extract_pbits(mode, bitblock)

        # If P-bit is shared
        if mode == 1:
            pbit_zero = pbits & 0x1  # Shared for first endpoint set
            pbit_one = pbits >> 1  # Shared for second endpoint set
            
            # rgb component-wise insert pbits
            for i in range(len(endpoint_array)):
                pbit = pbit_one if i >> 1 else pbit_zero
                for j in range(3):
                    endpoint_array[i][j] |= pbit 

        else:  # unique P-bit per endpoint, applied for RGBA
            for i in range(len(endpoint_array)):
                pbit = (pbits >> i) & 0x1
                for j in range(4):
                    endpoint_array[i][j] |= pbit

    for i in range(len(endpoint_array)):
        # Colour_component_precision & alpha_component_precision includes pbit
        # left shift endpoint components so that their MSB lies in bit 7, then
        # replicate each component's MSB into the LSBs revealed by the left-shift
        for j in range(3):
            endpoint_array[i][j] = endpoint_array[i][j] << (8 - get_colour_bits(mode))
            endpoint_array[i][j] |= endpoint_array[i][j] >> get_colour_bits(mode)
        if mode <= 3:  # Modes 0, 1, 2 and 3 are all fully opaque
            endpoint_array[i][3] = 255
        else:
            endpoint_array[i][3] = endpoint_array[i][3] << (8 - get_alpha_bits(mode))
            endpoint_array[i][3] |= endpoint_array[i][3] >> get_alpha_bits(mode)

    return endpoint_array

def decompress_bc7(x: int, y: int, block: bytes):
    """Try to get the pixel value out of a block of BC7 bytes."""
    bitblock = int.from_bytes(block, byteorder="big")
    mode = extract_mode(bitblock)
    
    # decode partition data from explicit partition bits
    subset_index = 0
    num_subsets = 1
    
    if mode in [0, 1, 2, 3, 7]:
        num_subsets = get_num_subsets(mode)
        partition_set_id = extract_partition_set_id(mode, bitblock)
        subset_index = get_partition_index(num_subsets, partition_set_id, x, y)
    
    # Extract raw, compressed endpoint bits
    endpoint_array = extract_endpoints(mode, bitblock)
    
    # Decode endpoint color and alpha for each subset
    endpoint_array = fully_decode_endpoints(endpoint_array, mode, bitblock)
    
    # Endpoints are now complete.
    endpoint_start[4] = endpoint_array[2 * subset_index]
    endpoint_end[4]   = endpoint_array[2 * subset_index + 1]
        
    # Determine the palette index for this pixel
    alpha_index     = get_alpha_index(block, mode, x, y);
    alpha_bitcount  = get_alpha_bitcount(block, mode);
    color_index     = get_color_index(block, mode, x, y);
    color_bitcount  = get_color_bitcount(block, mode);

    # Determine output
    output = bytes([0, 0, 0, 0])
    # output.rgb = interpolate(endpoint_start.rgb, endpoint_end.rgb, color_index, color_bitcount);
    # output.a   = interpolate(endpoint_start.a,   endpoint_end.a,   alpha_index, alpha_bitcount);
    
    if mode in [4, 5]:
        # Decode the 2 color rotation bits as follows:
        # 00 – Block format is Scalar(A) Vector(RGB) - no swapping
        # 01 – Block format is Scalar(R) Vector(AGB) - swap A and R
        # 10 – Block format is Scalar(G) Vector(RAB) - swap A and G
        # 11 - Block format is Scalar(B) Vector(RGA) - swap A and B
        rotation = extract_rot_bits(mode, block)
        output = swap_channels(output, rotation)

    return output

int decode_bc7(const uint8_t* data, uint32_t m_width, uint32_t m_height, uint32_t* image) {
	uint32_t m_block_width = 4;
	uint32_t m_block_height = 4;
	uint32_t m_blocks_x = (m_width + m_block_width - 1) / m_block_width;
	uint32_t m_blocks_y = (m_height + m_block_height - 1) / m_block_height;
	uint32_t buffer[16];
	for (uint32_t by = 0; by < m_blocks_y; by++) {
		for (uint32_t bx = 0; bx < m_blocks_x; bx++, data += 16) {
			decode_bc7_block(data, buffer);
			copy_block_buffer(bx, by, m_width, m_height, m_block_width, m_block_height, buffer, image);
		}
	}
	return 1;
}

bitblock = int.from_bytes(dds_bytestream[16:32], byteorder="big")
print(f"{bitblock:032X}")
# print(decompress_bc7(0, 0, dds_bytestream[:16]))

In [ ]:
# (uint)(-(int)(z & 1))

z = [0, 1, 2, 3, 257]

for x in z:
    print(-(x & 1))